# Exercises XP — LLM Fundamentals

**Course:** Developers Institute  **Week 7 - Day 2**  
**Author:** Alex Goldbaum

Six exercises covering: traditional vs modern NLP, BERT/GPT/T5 architectures,
pre-training benefits + ethics, transformer internals (self-attention,
multi-head, MLM/CLM, model selection, positional encoding), BERT variations,
and softmax temperature with a small executable demo at the end.


## Exercise 1 — Traditional vs Modern NLP

### 1.1 Comparative Table

| Aspect | Traditional NLP | Modern NLP (Transformers / LLMs) |
|---|---|---|
| **Feature engineering** | **Manual** — analysts design TF-IDF, n-grams, POS tags, regex rules. | **Automatic** — the model learns features end-to-end from raw text. |
| **Word representations** | **Static**: bag-of-words, one-hot, or fixed word2vec/GloVe vectors (the same vector for *bank* whether it is a river or a financial institution). | **Contextual**: each word's vector depends on the surrounding sentence (BERT, GPT embeddings shift with context). |
| **Model architectures** | **Shallow** — Naïve Bayes, SVM, logistic regression, CRF, sometimes small LSTM/RNN. | **Deep**: many-layer Transformer encoders/decoders with hundreds of millions to billions of parameters. |
| **Training methodology** | **Task-specific** — every task trained from scratch on its own labeled dataset. | **Pre-training + fine-tuning** — one giant self-supervised model is reused for many downstream tasks with little extra labeled data. |
| **Key model examples** | Naïve Bayes, SVM (text classification), HMM (POS tagging), CRF (NER), word2vec, GloVe, BiLSTM-CRF. | BERT, RoBERTa, DistilBERT, GPT-2/3/4, T5, FLAN-T5, LLaMA, Mistral, Claude. |
| **Advantages** | Cheap to train, easy to interpret, small models, low latency, no GPU required, very low data needs once features are good. | State-of-the-art accuracy, transfer learning across tasks/languages, zero/few-shot capabilities, captures syntax and semantics jointly. |
| **Disadvantages** | Heavy human feature engineering, brittle to domain shift, struggles with long-range dependencies and disambiguation. | Massive compute and data requirements, training cost in dollars, opaque decisions, hallucinations, bias risks at scale. |

### 1.2 Impact on scalability and efficiency

The shift from traditional to modern NLP has been a structural change, not an
incremental one. Three concrete effects:

- **Single model, many tasks.** A pre-trained backbone (BERT, T5) replaces dozens
  of bespoke pipelines. Adding a new task means fine-tuning a small head on a
  small labeled set, not building a new feature engineering pipeline from
  scratch. Engineering throughput goes up by an order of magnitude.
- **Multilingual scale.** Multilingual encoders (mBERT, XLM-RoBERTa) cover 100+
  languages with one weight checkpoint, where traditional NLP required one
  pipeline per language.
- **Efficiency cost has shifted.** The compute bill exploded — training a GPT-
  scale model costs millions — but **per-query** efficiency at inference has
  improved through distillation (DistilBERT), quantization, and KV caching.
  Operationally, modern NLP is *more* compute-hungry but *also* far more
  reusable: the same checkpoint serves classification, retrieval, generation,
  and translation.

The trade-off is now between **raw model performance** and **cost / latency /
interpretability**. Traditional methods are still the right default for small
datasets, strict latency budgets, and regulated environments where every
feature must be explainable.


## Exercise 2 — LLM Architecture and Application Scenarios

### BERT (Encoder-only, bidirectional, Masked Language Modeling)
**Architecture.** A stack of Transformer **encoders** — every token attends to
*every other* token in the input (bidirectional attention). Pre-training
objective is **MLM**: 15% of input tokens are masked, the model predicts them
from both left and right context. No autoregressive generation.

**Real-world application: sentiment classification of customer reviews.**
BERT excels because it reads the *whole* review at once before producing a
single class label. To label `"The food was great but the service was awful"`
the model needs to weigh both halves of the sentence — bidirectional
attention sees them simultaneously. Generation is not needed; we want one
label, which is exactly what an encoder + classification head produces.

---

### GPT (Decoder-only, unidirectional, Causal Language Modeling)
**Architecture.** A stack of Transformer **decoders** with **causal masking**:
every token only attends to tokens that came *before* it. Pre-training
objective is **CLM** (next-token prediction). Built for autoregressive
generation: at inference time the model produces one token at a time, each
conditioned on everything generated so far.

**Real-world application: open-ended chat / creative writing assistant.**
GPT-style models excel here because chat is fundamentally generative — we
want fluent, coherent, long-form text. Causal LM training matches the
inference pattern exactly (predict the next token), so the model is built for
the job: planning a multi-sentence answer is the same operation it was
trained to do.

---

### T5 (Encoder-Decoder, text-to-text)
**Architecture.** A full **encoder + decoder** Transformer. The encoder reads
the input bidirectionally; the decoder generates the output token by token
while cross-attending to the encoder. T5 frames *every* task as
`input text → output text` (translation, summarization, classification, QA).

**Real-world application: English → Spanish machine translation.**
Encoder-decoder is the canonical architecture for sequence-to-sequence
transformations — input and output are both sequences of varying length,
with different vocabularies and structures. The encoder builds a full
representation of the English source, the decoder generates Spanish
conditioned on it via cross-attention. T5 was trained on this kind of
text-to-text reformulation, so translation is essentially its natural shape.


## Exercise 3 — Benefits and Ethics of Pre-Training

### 3.1 Five key benefits of pre-trained models

1. **Improved generalization.** Pre-training on billions of tokens exposes the
   model to virtually every linguistic pattern, so it generalizes far better
   than a model trained from scratch on a small downstream dataset.
2. **Reduced need for labeled data.** Pre-training is *self-supervised* (no
   human labels needed). Fine-tuning afterwards often works with thousands of
   labeled examples instead of millions. This unlocks NLP for low-resource
   domains.
3. **Faster fine-tuning.** Adapting a pre-trained model to a new task converges
   in minutes-to-hours instead of the days a from-scratch model would need.
4. **Transfer learning.** A single pre-trained backbone serves dozens of tasks
   (classification, NER, QA, summarization). The same weights are reused, with
   only the task-specific head changing.
5. **Robustness.** Pre-trained models handle typos, paraphrases, unseen
   vocabulary and domain shift much better than narrow task-specific models
   because they have already seen many surface variations during pre-training.

### 3.2 Ethical concerns

- **Bias amplification.** Web-scale corpora encode social, gender and racial
  biases; the model learns and *amplifies* them. A pre-trained model used in
  hiring screening, credit, or healthcare can entrench discrimination at scale.
- **Misinformation and hallucinations.** LLMs confidently produce plausible-
  sounding but wrong content. Deployed without guardrails (e.g., in education,
  medical advice, news) this directly harms users.
- **Privacy leakage.** Training data may include personal information that
  the model can be coaxed into repeating verbatim.
- **Misuse for harmful content.** Phishing, scaled disinformation, deepfakes,
  fraud automation — capabilities like fluent text generation lower the cost
  of attacks.
- **Environmental and economic cost.** Training a single large model emits
  significant CO₂ and concentrates power among a few well-funded actors.

### 3.3 Mitigation strategies

- **Data curation and documentation** (datasheets, model cards) — know what
  the model was trained on and disclose it.
- **Debiasing techniques**: counterfactual data augmentation, balanced fine-
  tuning, fairness-aware loss functions, audits across demographic subgroups.
- **Output safety**: RLHF, content moderation classifiers, refusal training,
  rate limiting, attribution and source-citation in RAG setups.
- **Watermarking and provenance** for generated content.
- **Smaller / distilled models** (DistilBERT, TinyLlama) when full LLMs are not
  needed, to reduce energy use and concentration of power.
- **Independent audits and red-teaming** before deployment in sensitive domains.
- **Clear human-in-the-loop policies** for any high-stakes decision.


## Exercise 4 — Transformer Deep Dive

### 4.1 Self-Attention and Multi-Head Attention

**How self-attention works.** Each token is projected into three vectors —
**Q** (query), **K** (key) and **V** (value) — via three learned linear
transformations. The attention score from token *i* to token *j* is the
scaled dot product `softmax(Q_i · K_j / √d_k)`. The output for position *i*
is the weighted sum `Σ_j attention(i, j) · V_j`. In a sentence, every token
thus pools information from every other token, weighted by their semantic
compatibility. This is *self*-attention because the same sequence supplies
Q, K and V.

**Multi-head attention.** Instead of one attention computation, the model
runs **H parallel attention heads** (typically 8 or 12). Each head has its
own Q/K/V projections — so each head can specialize in a different *kind*
of relationship: one head can track syntactic dependencies, another
coreference, another long-range topical similarity, etc. The outputs are
concatenated and projected back to the model dimension. The advantage over
single-head: the model can represent **multiple relational structures
simultaneously** instead of forcing all of them into one attention map.

**Concrete example.** Take the sentence:

> *"The chef who studied in Paris cooked us a meal we will not forget."*

Different heads might do different things:
- A *syntactic* head links `cooked` with its subject `chef` (across the long
  relative clause).
- A *coreference* head links `we` with the implicit reader/customer pronoun chain.
- A *modifier* head links `meal` with `forget` (negated → memorable).
- A *named-entity* head highlights `Paris` and tags it as a location modifier of `chef`.

Each head extracts a different slice of the sentence structure; the next
layer combines them.

---

### 4.2 MLM vs CLM

| | **Masked Language Modeling** | **Causal Language Modeling** |
|---|---|---|
| Training task | Predict masked tokens using **both** left and right context. | Predict the **next** token given only the left context. |
| Attention | Bidirectional. | Causal (lower-triangular mask). |
| Used in | BERT, RoBERTa, DistilBERT, encoder side of T5/BART. | GPT family, LLaMA, Mistral, decoder side of T5/BART. |
| Output | Single dense representation of input (good for classification, NER, QA). | Autoregressive token sequence (good for generation). |

**When MLM is more appropriate.** Document classification, sentiment analysis,
named entity recognition — any task where we need a **rich understanding of
an already-given input** and we produce a single label or span.

**When CLM is more appropriate.** Chatbots, story generation, code
completion, summarization, translation — any task where the output is a
*generated sequence* conditioned on a prefix.

**Why early BERT used Next Sentence Prediction (NSP).** Original BERT added an
NSP objective that fed two sentences and asked the model to predict if the
second naturally follows the first. The intuition was that this would help
downstream tasks involving sentence pairs (NLI, QA).

**Why modern models avoid NSP.** Follow-up work (RoBERTa, ALBERT) showed NSP
either does not help or even hurts: the task is too easy (two random
sentences are usually obviously unrelated) and it does not actually train
deep cross-sentence reasoning. Modern training drops NSP and uses longer
contiguous spans with MLM only, which improved benchmark performance.

---

### 4.3 Transformer model selection

**1. Sentiment classifier for customer reviews.** ➜ **Encoder-only**
(BERT, RoBERTa, DistilBERT).
*Why:* the task is classification, not generation. We need a bidirectional
representation of the full review to produce a single label. A decoder would
be over-kill (and slower); an encoder-decoder would waste compute on a
decoder we do not need.

**2. Open-ended creative chatbot.** ➜ **Decoder-only**
(GPT-style models: GPT-4, LLaMA, Mistral).
*Why:* the system *generates* free-form responses token by token. Causal
language modeling matches the inference pattern. Decoder-only models are
trained at huge scale, scale up cleanly, and benefit from KV-caching for
low-latency streaming.

**3. English ➜ Spanish technical document translation.** ➜ **Encoder-Decoder**
(T5, mBART, NLLB).
*Why:* this is the canonical sequence-to-sequence task. The encoder builds a
full bidirectional representation of the English source; the decoder
generates Spanish while cross-attending to that representation. Decoder-only
models can also translate but encoder-decoder is the natural fit and
typically gives better fidelity, especially for long technical inputs.

---

### 4.4 Positional Encoding

**Purpose.** The self-attention operation is **permutation-invariant** — by
itself it has no notion of token order. Positional encoding injects
information about a token's position in the sequence so the model can
distinguish word order. Two common forms: fixed sinusoidal vectors (original
Transformer) and learned positional embeddings (BERT). Modern models also
use rotary (RoPE) or ALiBi-style biases.

**A failure case without positional encoding.**

> *"The dog chased the cat"* vs *"The cat chased the dog"*

Both sentences contain exactly the same word multiset. Without positional
information, self-attention would treat them as the same input and produce
the same representation — disastrous for tasks like translation or
sentiment, where word order changes meaning completely.


## Exercise 5 — BERT Variations: Choose Your Detective

### 5.1 Scenario-by-scenario recommendation

**Scenario 1 — Real-time sentiment on a resource-constrained mobile app.**
→ **DistilBERT.** Smallest of the BERT family (~60% of BERT-base, with ~97%
of its accuracy on GLUE). Knowledge distillation removes layers and
parameters, giving fast inference and small memory footprint. Perfect for
edge / mobile.

**Scenario 2 — Research on legal documents requiring high accuracy.**
→ **RoBERTa** (or a domain-pretrained variant like Legal-BERT). RoBERTa is
BERT with a much larger pre-training corpus, dynamic masking, longer
training and no NSP — it is BERT's accuracy-maxed cousin. When accuracy is
more important than cost, RoBERTa wins.

**Scenario 3 — Global customer support in many languages.**
→ **XLM-RoBERTa.** Trained on 100+ languages with the RoBERTa recipe. One
model handles English, Spanish, Japanese, Arabic, etc., which is exactly
what a multilingual customer support system needs. Avoids maintaining one
model per language.

**Scenario 4 — Efficient pre-training with replaced-token detection.**
→ **ELECTRA.** Instead of predicting masked tokens, ELECTRA trains a
discriminator to detect which tokens have been *replaced* by a small
generator. This makes every input token informative (vs only the 15% masked
in MLM), so ELECTRA reaches the same accuracy with **~25% of the pre-training
compute**.

**Scenario 5 — Efficient NLP in resource-constrained environments**
(e.g., a low-RAM server or edge device that still needs strong accuracy).
→ **ALBERT**. Uses cross-layer parameter sharing and factorized embedding
matrices to dramatically reduce parameter count while keeping accuracy
competitive. Trains and serves cheaper than BERT.

### 5.2 BERT variations comparison table

| Variant | Training data + method | Size / efficiency | Key optimization | Ideal use case |
|---|---|---|---|---|
| **BERT-base** | BookCorpus + English Wikipedia (~3.3 B tokens), MLM + NSP. | 110 M params. Baseline. | Bidirectional MLM + NSP. | Generic NLP baseline. |
| **RoBERTa** | 10× more text (CC-News, OpenWebText, Stories), MLM only with **dynamic masking**, much longer training, large batches. | ~125 M params, same size as BERT-base but trained much harder. | Drop NSP, larger batches, longer training. | High-accuracy NLU, research, anything where compute is available. |
| **ALBERT** | Same as BERT, with **cross-layer parameter sharing** and **factorized embeddings**. Uses Sentence-Order Prediction instead of NSP. | ~12-18 M params at base size (huge reduction). Faster, smaller. | Parameter sharing across layers. | Resource-constrained accuracy, mid-sized servers. |
| **DistilBERT** | Knowledge distillation from BERT-base. | ~66 M params, ~60% the size, ~60% faster at inference, retains ~97% of GLUE. | Distillation + triple loss (LM + cosine + soft labels). | Mobile/edge, real-time apps, latency-critical APIs. |
| **ELECTRA** | Replaced-token detection on the full BERT corpus. | Same size class as BERT but trains in **~25% the compute**. | Discriminator + small generator, every token contributes to loss. | Efficient pretraining, fast convergence. |
| **XLM-RoBERTa** | 2.5 TB of CommonCrawl across **100 languages**. | 270 M (base) / 550 M (large) params. | Massive multilingual pretraining on RoBERTa recipe. | Multilingual NLP, low-resource languages. |


## Exercise 6 — Softmax Temperature: The Randomness Regulator

Softmax with temperature `T` is:

$$ p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}} $$

`T` controls how sharp or smooth the resulting distribution is.

### 6.1 Temperature scenarios

**T = 0.2 (sharp / cold).** Logits are amplified — the highest-scoring token
becomes nearly certain. The model is **deterministic and conservative**:
it almost always picks the single most likely word. Output is predictable,
repetitive, and on-topic — and prone to looping ("the the the") if the model
is uncertain.

**T = 1.5 (warm / soft).** The distribution flattens; less likely tokens have a
much better chance of being picked. Output is **creative and diverse**, but
also more incoherent, more error-prone, and more likely to drift off topic
or invent facts.

**T = 1.0 (neutral).** The softmax is unchanged from the model's native
distribution. This is the default for most chat models — the developers tuned
the model to produce reasonable behavior at this point.

### 6.2 Application design

**A: Personalized bedtime stories for children.** We want variety so the same
child never hears the exact same story twice, but **coherent and safe**
(no hallucinated scary content, no broken grammar). Use **T ≈ 0.8-1.1**:
warm enough to bring narrative variety, cold enough to stay on track. Layer
in **top-p (≈0.9)** or **top-k** filtering on top of temperature to forbid
very-low-probability tokens that could surface unsafe content. Pin
constraints with structured prompts (character names, setting).

**B: Auto-summary of financial reports.** Accuracy and reproducibility are
non-negotiable. Use **T ≤ 0.3** (often `T=0` if the API supports it) so the
same report always yields essentially the same summary. Combine with **RAG**
(retrieve the actual figures and quote them) to fight hallucinations.
Critically, **temperature is not a substitute for grounding**: a cold LLM
still hallucinates with confidence if the prompt does not include the facts.

### 6.3 Temperature and bias

Temperature interacts with bias in two ways:

- **At low T**: the model collapses onto its *most likely* completion. If
  the training distribution is biased (e.g., 'nurse' overwhelmingly followed
  by 'she'), low-T sampling will *consistently* surface that bias. The bias
  becomes more visible because alternatives are crowded out.
- **At high T**: the model samples from the long tail, which mitigates the
  most likely biased completion but also introduces *new* biases (rare,
  potentially toxic tokens become more accessible).

**Practical example.**
Prompt: *"The CEO walked into the room. The CEO is a ___"*
- At `T=0.1`, the model almost always completes with *"man"* — surfacing a
  statistical gender bias from the training data with high confidence.
- At `T=1.5`, the completions become more diverse (*"woman", "professional",
  "figure"*) but also less reliable.

**Bias is not solved by changing temperature.** Real mitigation requires data
curation, bias-aware fine-tuning (e.g., balanced counterfactual examples),
RLHF, and runtime safety filters. Temperature is a knob, not a fix.


## Quick Executable Demo — Softmax Temperature

To make Exercise 6 tangible, the cell below shows how the same logits
produce very different distributions at different temperatures.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Imaginary next-token logits for the prompt 'The weather today is ___'
vocab = ['sunny', 'cloudy', 'rainy', 'snowy', 'windy', 'cold', 'hot', 'fine', 'awful', 'mild']
logits = np.array([4.0, 3.5, 3.0, 1.5, 1.2, 2.5, 2.0, 2.8, 1.0, 1.8])


def softmax_temp(logits, T):
    z = logits / T
    z = z - z.max()  # numerical stability
    e = np.exp(z)
    return e / e.sum()


temperatures = [0.2, 1.0, 1.5]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, T in zip(axes, temperatures):
    probs = softmax_temp(logits, T)
    colors = sns.color_palette('viridis', n_colors=len(vocab))
    ax.bar(vocab, probs, color=colors, edgecolor='white')
    ax.set_title(f'T = {T}', fontweight='bold')
    ax.set_xticklabels(vocab, rotation=35, ha='right')
    ax.set_ylim(0, 1.0)
    for i, p in enumerate(probs):
        ax.text(i, p + 0.01, f'{p:.2f}', ha='center', fontsize=8)
axes[0].set_ylabel('Probability')
plt.suptitle('Same logits, three temperatures', fontweight='bold')
plt.tight_layout()
plt.show()

# Print top-3 at each temperature
for T in temperatures:
    p = softmax_temp(logits, T)
    order = p.argsort()[::-1][:3]
    top3 = [f"{vocab[i]}({p[i]:.2f})" for i in order]
    print(f'T={T}: top-3 = {top3}')


**What the demo shows.**
- At `T=0.2` one token dominates (probability close to 1) — sampling is
  essentially deterministic.
- At `T=1.0` the model's original distribution shows: a clear leader but
  meaningful probability mass on several alternatives.
- At `T=1.5` the distribution flattens further — more diversity, more risk of
  picking implausible tokens.

This is the single knob behind "cold" vs "creative" output in production
LLM APIs.


## Summary

- The shift from traditional to modern NLP is structural: one pre-trained
  Transformer replaces many bespoke pipelines, at the cost of much more
  compute.
- BERT, GPT and T5 split the work cleanly: **encoder-only** for understanding,
  **decoder-only** for generation, **encoder-decoder** for sequence-to-sequence.
- Pre-training delivers concrete benefits (generalization, transfer, less
  labeled data) but introduces real risks (bias, hallucination, privacy,
  misuse, environmental cost) that need active mitigation.
- Self-attention + multi-head + positional encoding make the Transformer
  capture both local and long-range structure. MLM trains for understanding,
  CLM trains for generation, and modern recipes drop NSP.
- The BERT family (RoBERTa, ALBERT, DistilBERT, ELECTRA, XLM-RoBERTa) trades
  accuracy, speed, size and multilingual coverage in different ways — pick the
  one whose trade-off matches your deployment.
- Softmax temperature is the simplest knob to control creativity vs
  determinism in generation. Use it explicitly, but do not rely on it to fix
  bias or hallucination.
